# Mouse Brain Spatial Multi-omics

## Setup

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import scanpy as sc
import torch

import SpaDiff as sd
from SpaDiff.utils import set_seed

In [ ]:
SEED = 42
N_LATENT = 50
N_NEIGHBORS = 7
TRAINING_EPOCHS = 500

DATA_ROOT = Path("path/Mouse_Brain")
RNA_FILE = DATA_ROOT / "RNA.h5ad"
ATAC_FILE = DATA_ROOT / "ATAC.h5ad"

set_seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## Paired data

In [ ]:
adata_rna = sc.read_h5ad(RNA_FILE)
adata_atac = sc.read_h5ad(ATAC_FILE)
adata_rna.var_names_make_unique()
adata_atac.var_names_make_unique()

common_spots = adata_rna.obs_names[adata_rna.obs_names.isin(adata_atac.obs_names)]
adata_rna = adata_rna[common_spots].copy()
adata_atac = adata_atac[common_spots].copy()

rna_spatial = np.asarray(adata_rna.obsm["spatial"], dtype=np.float64)

## Preprocessing

In [ ]:
sc.pp.filter_genes(adata_rna, min_cells=10)
adata_rna.layers["counts"] = adata_rna.X.copy()
sc.pp.normalize_total(adata_rna, target_sum=1e4)
sc.pp.log1p(adata_rna)
sc.pp.highly_variable_genes(adata_rna, flavor="seurat", n_top_genes=3000)
adata_rna = adata_rna[:, adata_rna.var["highly_variable"]].copy()
sc.pp.scale(adata_rna)
sc.tl.pca(adata_rna, n_comps=N_LATENT, svd_solver="arpack", random_state=SEED)

X_atac_lsi_np, _ = sd.atac_lsi(
    adata_atac,
    n_components=N_LATENT,
    min_cells=10,
    random_state=SEED,
)

X_rna_np = np.asarray(adata_rna.obsm["X_pca"], dtype=np.float32)
X_atac_np = np.asarray(X_atac_lsi_np, dtype=np.float32)

adata_rna.obsm["X_rna_pca"] = X_rna_np
adata_rna.obsm["X_atac_lsi"] = X_atac_np
adata_atac.obsm["X_lsi"] = X_atac_np

## Shared spatial topology

In [ ]:
spatial_topology = sd.build_spatial_topology(
    adata_rna,
    mode="global_knn",
    n_neighbors=N_NEIGHBORS,
    max_order=2,
    device=device,
)
base_operators = spatial_topology.operators

## Paired-modality SpaDiff training

A shared-parameter model learns separate $H^{RNA}$ and $H^{ATAC}$ representations with an explicit modality condition. The joint representation is their spot-wise arithmetic mean.

In [ ]:
config = sd.SpaDiffConfig(
    data_dim=N_LATENT,
    condition_input_dim=N_LATENT,
    num_modalities=2,
)
model = sd.SpaDiff(config).to(device)
adata_joint = model.fit_transform_multiomics(
    adata_rna,
    adata_atac,
    X_rna_np,
    X_atac_np,
    base_operators,
    epochs=TRAINING_EPOCHS
)

## Spatial domains

In [ ]:
sc.pp.neighbors(adata_joint, use_rep="spadiff_joint", random_state=SEED,)
sc.tl.louvain(adata_joint, random_state=SEED)

In [ ]:
adata_joint.obsm["spatial"] = np.column_stack((-rna_spatial[:, 1], rna_spatial[:, 0]))
sc.pl.spatial(adata_joint, color="louvain", spot_size=1)